In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import pandas as pd
import glob

# 找到所有CSV文件
path = '/content/drive/MyDrive/idx_dataset/'
files = glob.glob(path + 'CRMLSSold*.csv')

# 合并成一个dataframe
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(df.shape)
df.head()

/tmp/ipykernel_6034/1928895754.py:9: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
/tmp/ipykernel_6034/1928895754.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


(189608, 78)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,LotSizeDimensions,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict
0,NorthSanLuisObispo,NorthSanLuisObispo,NaN,NaN,NaN,NaN,NaN,NaN,479293596,newlin@pacificaCRE.com,...,NaN,1807.0,NaN,NaN,NaN,NaN,93446,NaN,1807.0,NaN
1,TheInlandGateway,TheInlandGateway,NaN,True,NaN,NaN,NaN,12000.0,446914808,nleimers@hotmail.com,...,NaN,14253.0,NaN,False,NaN,NaN,92325,0.0,14253.0,NaN
2,SanDiego,SanDiego,Wood,True,NaN,NaN,False,1695.0,445176588,mannybehar@yahoo.com,...,NaN,NaN,NaN,NaN,0.0,NaN,92126,0.0,NaN,NaN
3,SanDiego,SanDiego,NaN,False,NaN,NaN,False,950000.0,1145282039,chase@cromwellhomegroup.com,...,NaN,NaN,NaN,False,2.0,NaN,91901,NaN,NaN,NaN
4,SanDiego,SanDiego,NaN,False,NaN,NaN,False,970000.0,1145278097,rory@corneliusestates.net,...,NaN,NaN,NaN,False,2.0,NaN,92115,NaN,NaN,NaN


In [15]:
import matplotlib.pyplot as plt
import seaborn as sns

df_clean = df[
    (df['ClosePrice'] > 100000) & (df['ClosePrice'] < 5000000) &
    (df['LivingArea'] > 200) & (df['LivingArea'] < 10000) &
    (df['LotSizeSquareFeet'] > 100) & (df['LotSizeSquareFeet'] < 100000)
]
print(f'去掉异常值后数据量: {df_clean.shape}')

去掉异常值后数据量: (107284, 78)


In [16]:
# 统计每列的缺失比例
missing_pct = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)
missing_summary = pd.DataFrame({
    'missing_count': df_clean.isnull().sum(),
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

print(missing_summary[missing_summary['missing_pct'] > 0])

                              missing_count  missing_pct
AboveGradeFinishedArea               107284   100.000000
TaxYear                              107284   100.000000
MiddleOrJuniorSchoolDistrict         107284   100.000000
FireplacesTotal                      107284   100.000000
ElementarySchoolDistrict             107284   100.000000
CoveredSpaces                        107284   100.000000
BusinessType                         107282    99.998136
WaterfrontYN                         107208    99.929160
TaxAnnualAmount                      107055    99.786548
BelowGradeFinishedArea               106561    99.326088
BasementYN                           105089    97.954029
BuilderName                          102174    95.236941
LotSizeDimensions                    101284    94.407367
BuildingAreaTotal                     98335    91.658588
CoBuyerAgentFirstName                 97149    90.553111
ElementarySchool                      93754    87.388613
MiddleOrJuniorSchool           

In [17]:
# ============================================================
# Handle missing values in df_clean
# Strategy: drop fully-empty columns, flag boolean/co-agent
# fields where NaN means "not applicable", impute numeric
# fields with median, fill categorical/text fields with
# 'Unknown', and drop rows for low-missing critical fields
# ============================================================

# Drop columns that are 100% (or effectively 100%) missing —
# these carry no usable information at all
drop_cols_full = [
    'AboveGradeFinishedArea', 'TaxYear', 'MiddleOrJuniorSchoolDistrict',
    'FireplacesTotal', 'ElementarySchoolDistrict', 'CoveredSpaces', 'BusinessType'
]
df_clean = df_clean.drop(columns=drop_cols_full)

# Boolean feature flags: NaN here means the property simply doesn't
# have that feature (e.g. no waterfront, no basement), not missing data
# (explicitly infer_objects to avoid FutureWarning on downcasting)
bool_flag_cols = ['WaterfrontYN', 'BasementYN', 'AttachedGarageYN',
                   'NewConstructionYN', 'PoolPrivateYN', 'ViewYN', 'FireplaceYN']
for col in bool_flag_cols:
    df_clean[col] = df_clean[col].fillna(False).infer_objects(copy=False)

# Co-agent fields: NaN means there was no co-agent on the deal,
# so fill with an explicit 'None' label rather than dropping
co_agent_cols = ['CoBuyerAgentFirstName', 'CoListAgentFirstName',
                  'CoListAgentLastName', 'CoListOfficeName']
for col in co_agent_cols:
    df_clean[col] = df_clean[col].fillna('None')

# Very high missing-rate numeric/text columns (>90%) with no clear
# "missing = 0/None" interpretation — drop entirely to avoid noisy imputation
drop_cols_high = ['TaxAnnualAmount', 'BelowGradeFinishedArea', 'BuilderName',
                   'LotSizeDimensions', 'BuildingAreaTotal']
df_clean = df_clean.drop(columns=drop_cols_high)

# School-related fields: missing likely means the MLS listing just
# didn't fill it in, not that no school exists — fill as 'Unknown'
school_cols = ['ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool', 'HighSchoolDistrict']
for col in school_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')

# HOA-related fields: missing AssociationFee most likely means
# there is no HOA, so fill with 0 / 'None' instead of imputing
df_clean['AssociationFee'] = df_clean['AssociationFee'].fillna(0)
df_clean['AssociationFeeFrequency'] = df_clean['AssociationFeeFrequency'].fillna('None')

# Numeric structural features with low-to-moderate missingness —
# impute with median to preserve row count without heavy skew
# (Stories/Levels excluded — they are categorical text, not numeric)
numeric_impute_cols = ['MainLevelBedrooms', 'GarageSpaces',
                        'BedroomsTotal', 'BathroomsTotalInteger']
for col in numeric_impute_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Categorical/text descriptive fields — fill with 'Unknown' to
# preserve rows while flagging the missingness explicitly
# (Stories and Levels included here since they hold text values
# like 'ThreeOrMore' / 'Two,MultiSplit' rather than numbers)
categorical_fill_cols = ['Flooring', 'MLSAreaMajor', 'SubdivisionName',
                          'PropertySubType', 'Stories', 'Levels']
for col in categorical_fill_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')

# Agent/office identifying fields with low missing rates —
# fill with 'Unknown' rather than dropping rows over non-critical metadata
low_missing_cols = ['ListAgentFirstName', 'BuyerAgentFirstName', 'ListAgentEmail',
                     'BuyerAgentMlsId', 'BuyerAgentLastName', 'ListAgentFullName',
                     'ListAgentAOR', 'ListAgentLastName', 'BuyerOfficeName', 'BuyerAgentAOR',
                     'BuyerOfficeAOR']
for col in low_missing_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')

# Critical fields (location, price, date, address) with very low
# missing rates (<1%) — drop the affected rows since these are
# core features/join keys and imputing them would be unreliable
critical_cols = ['YearBuilt', 'City', 'Longitude', 'Latitude', 'PurchaseContractDate',
                  'PostalCode', 'ParkingTotal', 'OriginalListPrice',
                  'StreetNumberNumeric', 'UnparsedAddress']
df_clean = df_clean.dropna(subset=critical_cols)

# Check final shape after all missing-value handling steps
print(f"Shape after handling missing values: {df_clean.shape}")

/tmp/ipykernel_6034/3290801471.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean[col] = df_clean[col].fillna(False).infer_objects(copy=False)


Shape after handling missing values: (106857, 66)


In [18]:
# Identify all object/categorical columns remaining in df_clean,
# along with their cardinality (number of unique values), to decide
# the appropriate encoding strategy for each
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
cardinality = df_clean[categorical_cols].nunique().sort_values(ascending=False)
print(cardinality)

ListingId                   106794
UnparsedAddress             106108
BuyerAgentMlsId              53721
ListAgentEmail               41664
ListAgentFullName            40642
BuyerAgentLastName           23470
ListAgentLastName            20659
BuyerAgentFirstName          11585
BuyerOfficeName              11148
ListOfficeName               10172
ListAgentFirstName            9273
SubdivisionName               8114
CoListAgentLastName           7195
CoListAgentFirstName          3632
CoListOfficeName              2916
CoBuyerAgentFirstName         2312
PostalCode                    1903
ElementarySchool              1195
MLSAreaMajor                   977
City                           896
ListingContractDate            842
MiddleOrJuniorSchool           607
PurchaseContractDate           561
HighSchool                     486
HighSchoolDistrict             405
ContractStatusChangeDate       272
CloseDate                      272
Flooring                       257
CountyOrParish      

In [19]:
# ============================================================
# Convert categorical fields to numeric (encoding) — revised
# Flooring, CountyOrParish, and the three AOR fields are moved
# from one-hot to frequency encoding, since their high cardinality
# (256, 59, 57, 55, 54 categories respectively) caused excessive
# dimensionality blow-up with limited modeling benefit
# ============================================================

# Pure identifier / free-text fields — no generalizable modeling
# signal (unique IDs, emails, raw addresses), drop entirely
id_text_cols = [
    'ListingId', 'UnparsedAddress', 'BuyerAgentMlsId', 'ListAgentEmail',
    'ListAgentFullName', 'BuyerAgentLastName', 'ListAgentLastName',
    'BuyerAgentFirstName', 'ListAgentFirstName', 'CoListAgentLastName',
    'CoListAgentFirstName', 'CoBuyerAgentFirstName'
]
df_clean = df_clean.drop(columns=id_text_cols, errors='ignore')

# MlsStatus has only 1 unique value — zero variance, drop it
df_clean = df_clean.drop(columns=['MlsStatus'], errors='ignore')

# Date fields — convert to datetime, extract year/month as numeric
# components instead of treating them as categorical text
date_cols = ['ListingContractDate', 'PurchaseContractDate',
             'ContractStatusChangeDate', 'CloseDate']
for col in date_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
        df_clean[f'{col}_year'] = df_clean[col].dt.year
        df_clean[f'{col}_month'] = df_clean[col].dt.month
        df_clean = df_clean.drop(columns=[col])

# High-cardinality categorical fields — one-hot would explode
# dimensionality, so use frequency encoding (replace category with
# its occurrence count) instead. Flooring, CountyOrParish, and the
# AOR fields are included here since each has 50+ unique categories
high_card_cols = [
    'BuyerOfficeName', 'ListOfficeName', 'SubdivisionName', 'CoListOfficeName',
    'PostalCode', 'ElementarySchool', 'MLSAreaMajor', 'City',
    'MiddleOrJuniorSchool', 'HighSchool', 'HighSchoolDistrict',
    'Flooring', 'CountyOrParish', 'BuyerOfficeAOR', 'BuyerAgentAOR', 'ListAgentAOR'
]
for col in high_card_cols:
    if col in df_clean.columns:
        freq_map = df_clean[col].value_counts()
        df_clean[f'{col}_freq'] = df_clean[col].map(freq_map)
        df_clean = df_clean.drop(columns=[col])

# Low cardinality categorical fields (fewer than ~20 categories) —
# safe to one-hot encode without excessive dimensionality
low_card_cols = [
    'PropertySubType', 'Levels', 'PropertyType',
    'AssociationFeeFrequency', 'StateOrProvince', 'Stories'
]
low_card_cols_present = [col for col in low_card_cols if col in df_clean.columns]
df_clean = pd.get_dummies(df_clean, columns=low_card_cols_present, drop_first=True)

# Boolean-type columns still stored as text after earlier fillna
# steps — convert to 0/1 integers
bool_cols = ['WaterfrontYN', 'BasementYN', 'AttachedGarageYN',
             'NewConstructionYN', 'PoolPrivateYN', 'ViewYN', 'FireplaceYN']
for col in bool_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(int)

# Confirm no object columns remain and check final shape
remaining_object_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f"Remaining object columns: {remaining_object_cols}")
print(f"Shape after encoding: {df_clean.shape}")

Remaining object columns: []
Shape after encoding: (106857, 100)


In [23]:
# ============================================================
# Step 1: Reconstruct a sortable year-month key from the CloseDate
# year/month columns (created earlier during categorical encoding)
# ============================================================
df_clean['CloseYearMonth'] = (
    df_clean['CloseDate_year'].astype(int) * 100 + df_clean['CloseDate_month'].astype(int)
)

# Sanity check: confirm which months are actually present in the data
print(f"Date range: {df_clean['CloseYearMonth'].min()} to {df_clean['CloseYearMonth'].max()}")
print(df_clean['CloseYearMonth'].value_counts().sort_index())

Date range: 202506 to 202605
CloseYearMonth
202506    13067
202510    13472
202511    10896
202512    11693
202601     8441
202602     9688
202603    12710
202604    13440
202605    13450
Name: count, dtype: int64


In [24]:
# ============================================================
# Step 2: Train/test split using ACTUAL CALENDAR MONTHS (not just
# "the N most recent available months"). This matters because the
# data has a gap (Jul-Sep 2025 have zero rows) — using calendar
# boundaries means a training window that spans the gap will
# naturally include fewer rows, rather than silently stretching
# across the missing months to hit a target row/month count.
#
# Test set  = most recent calendar month in the data
# Train set = the x_months of calendar time immediately preceding it
# ============================================================

def yearmonth_to_date(ym):
    """Convert an int YYYYMM into a (year, month) tuple."""
    return ym // 100, ym % 100

def add_months(year, month, delta):
    """Add delta months to a (year, month) pair, handling year rollover."""
    total = year * 12 + (month - 1) + delta
    return total // 12, total % 12 + 1

def get_train_test_split(df, x_months):
    all_months = sorted(df['CloseYearMonth'].unique())
    test_ym = all_months[-1]
    test_year, test_month = yearmonth_to_date(test_ym)

    # Calendar boundary x_months before the test month
    start_year, start_month = add_months(test_year, test_month, -x_months)
    start_ym = start_year * 100 + start_month
    end_ym = test_ym  # exclusive of the test month itself

    train_df = df[(df['CloseYearMonth'] >= start_ym) & (df['CloseYearMonth'] < end_ym)].copy()
    test_df = df[df['CloseYearMonth'] == test_ym].copy()

    return train_df, test_df

# Example usage with a 6-month training window
train_df, test_df = get_train_test_split(df_clean, x_months=6)
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
print(f"Train months present: {sorted(train_df['CloseYearMonth'].unique())}")
print(f"Test month: {sorted(test_df['CloseYearMonth'].unique())}")

Train shape: (66868, 101), Test shape: (13450, 101)
Train months present: [np.int64(202511), np.int64(202512), np.int64(202601), np.int64(202602), np.int64(202603), np.int64(202604)]
Test month: [np.int64(202605)]


In [25]:
# ============================================================
# Step 3: Normalize numerical features — fit StandardScaler on
# TRAIN ONLY, then transform both train and test. This prevents
# test-set statistics (mean/std) from leaking into the training
# process. Target column and date/id helper columns are excluded
# from scaling.
# ============================================================
from sklearn.preprocessing import StandardScaler

def normalize_features(train_df, test_df, target_col='ClosePrice', exclude_cols=None):
    """
    Scales numeric feature columns using a StandardScaler fit on train only.
    target_col and exclude_cols (e.g. date/id helper columns) are left untouched.
    """
    if exclude_cols is None:
        exclude_cols = []

    # Identify numeric feature columns to scale (exclude target + excluded cols)
    feature_cols = [
        c for c in train_df.columns
        if c not in exclude_cols + [target_col]
        and train_df[c].dtype in ['int64', 'float64', 'int32', 'float32']
    ]

    scaler = StandardScaler()
    train_scaled = train_df.copy()
    test_scaled = test_df.copy()

    train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_scaled[feature_cols] = scaler.transform(test_df[feature_cols])

    return train_scaled, test_scaled, scaler, feature_cols

# Columns that should NOT be scaled (target variable + date helper columns
# used only for splitting, not intended as model features)
exclude_cols = ['CloseYearMonth', 'CloseDate_year', 'CloseDate_month']

train_scaled, test_scaled, scaler, feature_cols = normalize_features(
    train_df, test_df, target_col='ClosePrice', exclude_cols=exclude_cols
)
print(f"Number of features scaled: {len(feature_cols)}")

Number of features scaled: 48


In [26]:
# 待确认：target_col='ClosePrice' — 如果目标变量列名不是这个，改一下参数。
# exclude_cols — 目前只排除了日期辅助列。如果feature_cols里还混进了其他不该进模型的列（比如ListingKey如果是数值型ID），需要加进exclude_cols，可以先跑print(feature_cols)看一眼确认。

In [27]:
from google.colab import files

df_clean.to_csv('cleaned_df.csv', index=False)
files.download('cleaned_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. 挂载Google Drive（如果还没挂载）
from google.colab import drive
drive.mount('/content/drive')

# 2. 输入token（不会保存到文件）
import getpass
token = getpass.getpass("输入你的GitHub Token: ")

# 3. Clone repo
%cd /content
!git clone https://{token}@github.com/rongweishen63-blip/idx-exchange-project.git
%cd idx-exchange-project

# 4. 复制notebook
!cp "/content/drive/MyDrive/Colab Notebooks/idx_project.ipynb" .

# 5. Push
!git config --global user.email "rongweishen63@gmail.com"
!git config --global user.name "rongweishen63-blip"

!cp "/content/drive/MyDrive/Colab Notebooks/02_preprocessing.ipynb" .

!git add .
!git commit -m "更新EDA"
!git push origin main

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# !git pull origin main --no-rebase